#### [Notebook 2 Quick Start](https://confluence.shopee.io/x/7O0lhw)
#### Common Use Cases and Solution Using Notebook 2
- [Multi Kernels in Notebook 2](https://confluence.shopee.io/x/4Od2hw)
- [Execute HDFS command](https://confluence.shopee.io/x/P5w8hw)
- [Browse, upload and download from Data Infra S3](https://confluence.shopee.io/x/xo9Ohw)
- [Setting up 3rd party packages for your Notebook 2](https://confluence.shopee.io/x/JZFOhw)
- [Develop and setup user defined function or library](https://confluence.shopee.io/x/hpjehg)
- [Execute Presto/Trino query directly in Python Kernel Notebook](https://confluence.shopee.io/x/WpMlhw)

> Note: "Python on K8S" Kernel cannot execute Spark functions. ***If you want to use Spark, change kernel to Spark***. [How to change Kernel](https://confluence.shopee.io/x/4Od2hw#MultiKernelsinNotebook2-SetKernelwhencreatingnewNotebook2file)

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
PySpark CPF个人资料处理脚本
将Open Finance个人数据处理成特征，用于风险评估

数据源：risk.dws_risk_br_openfinance_df
输出表：risk.ads_risk_br_openfinance_cpf_personal_profile_df
"""

import json
import sys
from datetime import datetime, date, timedelta, timezone
from typing import Dict, List, Any, Optional

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, udf, lit, when, isnan, isnull, to_json, from_json,
    explode, collect_list, struct, array, size, coalesce,
    to_timestamp, datediff, current_date, regexp_extract, broadcast
)
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    FloatType, BooleanType, MapType, ArrayType, DoubleType
)
from pyspark.sql import functions as F

import logging

# 配置日志
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# 定义特征字段常量
DEMOGR_ALL_FTS = [
    "inst_demogr_has_data",
    "inst_demogr_days_from_last_update",
    "inst_demogr_age",
    "inst_demogr_country"
]

INCOME_ALL_FTS = [
    "inst_income_has_data",
    "inst_income_days_from_last_update",
    "inst_income_currency",
    "inst_income_freq",
    "inst_income_amt",
    "inst_income_date"
]


# 将处理函数定义为独立函数，避免在UDF中引用SparkContext
def create_null_demogr_features() -> Dict[str, Any]:
    """创建人口统计特征的空值映射"""
    null_fts = {}
    for ft in DEMOGR_ALL_FTS:
        if ft == 'inst_demogr_has_data':
            null_fts[ft] = 0
        else:
            null_fts[ft] = None
    return null_fts


def create_null_income_features() -> Dict[str, Any]:
    """创建收入特征的空值映射"""
    null_fts = {}
    for ft in INCOME_ALL_FTS:
        if ft == 'inst_income_has_data':
            null_fts[ft] = 0
        else:
            null_fts[ft] = None
    return null_fts

from zoneinfo import ZoneInfo

def convert_to_timezone(dt_str: str):
    """
    将 ISO8601 格式时间字符串（UTC）转换为指定时区时间。
    
    参数:
        dt_str (str): 时间字符串，如 "2025-05-22T23:22:23Z" 或 "2025-05-22T23:22:23+00:00"
        target_tz (str): 目标时区（默认: America/Sao_Paulo）
        
    返回:
        datetime: 带时区信息的 datetime 对象
    """
    target_tz = "America/Sao_Paulo"
    # 自动解析常见格式
    dt_str = dt_str.strip()
    if dt_str.endswith("Z"):
        dt = datetime.strptime(dt_str, "%Y-%m-%dT%H:%M:%SZ").replace(tzinfo=timezone.utc)
    elif "+" in dt_str or "-" in dt_str[10:]:
        dt = datetime.fromisoformat(dt_str)
    else:
        raise ValueError(f"Unrecognized datetime format: {dt_str}")
    dt_local = dt.astimezone(ZoneInfo(target_tz))
    # 转换到目标时区
    return dt_local.strftime("%Y-%m-%d")



def parse_identifications(rsp_data: str, request_date: str) -> Dict[str, Any]:
    """
    解析身份信息

    Args:
        rsp_data: API响应数据JSON字符串
        request_date: 请求日期

    Returns:
        解析后的身份信息字典
    """
    if not rsp_data:
        return {}

    try:
        if isinstance(rsp_data, str):
            rsp_data = json.loads(rsp_data)

        # 处理数组格式的响应
        identification = rsp_data[0] if isinstance(rsp_data, list) else rsp_data
        if isinstance(identification, str):
            identification = json.loads(identification)

        return {
            'cpf': identification.get('documents', {}).get('cpfNumber', ''),
            'civil_name': identification.get('civilName', ''),
            'social_name': identification.get('socialName', ''),
            'birth_date': identification.get('birthDate', ''),
            'sex': identification.get('sex', ''),
            'marital_status': identification.get('maritalStatusCode', ''),
            'nationality': identification.get('hasBrazilianNationality', False),
            'brand_name': identification.get('brandName', ''),
            'personal_id': identification.get('personalId', ''),
            'update_datetime': convert_to_timezone(identification.get('updateDateTime', '')),
            'request_date': request_date
        }
    except Exception as e:
        # 在独立函数中无法直接使用logger，改为print
        print(f"解析身份信息时出错: {e}")
        return {}


def parse_qualifications(rsp_data: str, request_date: str) -> Dict[str, Any]:
    """
    解析资格信息

    Args:
        rsp_data: API响应数据JSON字符串
        request_date: 请求日期

    Returns:
        解析后的资格信息字典
    """
    if not rsp_data:
        return {}

    try:
        if isinstance(rsp_data, str):
            rsp_data = json.loads(rsp_data)

        # 提取收入信息
        income_info = rsp_data.get('informedIncome', {})
        amount_info = income_info.get('amount', {}) if income_info else {}

        informed_income = {
            'frequency': income_info.get('frequency', '') if income_info else '',
            'amount': amount_info.get('amount', '') if amount_info else '',
            'currency': amount_info.get('currency', '') if amount_info else '',
            'date': income_info.get('date', '') if income_info else ''
        }

        return {
            'company_cnpj': rsp_data.get('companyCnpj', ''),
            'occupation_code': rsp_data.get('occupationCode', ''),
            'occupation_description': rsp_data.get('occupationDescription', ''),
            'update_datetime': convert_to_timezone(rsp_data.get('updateDateTime', '')),
            'informed_income': informed_income,
            'request_date': request_date
        }
    except Exception as e:
        print(f"解析资格信息时出错: {e}")
        return {}


def extract_demogr_features(identifications: List[Dict]) -> Dict[str, Any]:
    """
    从身份信息中提取人口统计特征

    Args:
        identifications: 身份信息列表

    Returns:
        人口统计特征字典
    """
    if not identifications or len(identifications) == 0:
        return create_null_demogr_features()

    try:
        # 过滤有效数据
        valid_identifications = []
        for item in identifications:
            if item.get('birth_date') and item.get('update_datetime') and item.get('request_date'):
                has_data = 1
                birth_date = item['birth_date']
                update_time = item['update_datetime']
                nationality = item['nationality']
                request_date = item['request_date']

                # 计算更新时间差
                try:
                    # 计算更新时间差
                    fmt = "%Y-%m-%d"  # 根据你的实际格式调整
                    update_dt = datetime.strptime(update_time, fmt)
                    request_dt = datetime.strptime(request_date, fmt)

                    days_from_update = (request_dt - update_dt).days
                except:
                    days_from_update = None

                # 计算年龄
                try:
                    birth_dt = datetime.fromisoformat(birth_date)
                    age = int((datetime.now() - birth_dt).days / 365)
                except:
                    age = None
                if days_from_update is not None:
                    valid_identifications.append({
                        'has_data': has_data,
                        'days_from_last_update': days_from_update,
                        'age': age,
                        'nationality': nationality
                    })

        if not valid_identifications:
            return create_null_demogr_features()

        # 选择最近更新的记录
        latest_record = min(valid_identifications, key=lambda x: x['days_from_last_update'])

        return {
            "inst_demogr_has_data": latest_record['has_data'],
            "inst_demogr_days_from_last_update": latest_record['days_from_last_update'],
            "inst_demogr_age": latest_record['age'],
            "inst_demogr_country": 'BRA' if latest_record['nationality'] else 'OUTROS'
        }
    except Exception as e:
        print(f"提取人口统计特征时出错: {e}")
        return create_null_demogr_features()





def extract_income_features(qualifications: List[Dict]) -> Dict[str, Any]:
    """
    从资格信息中提取收入特征

    Args:
        qualifications: 资格信息列表

    Returns:
        收入特征字典
    """
    if not qualifications or len(qualifications) == 0:
        return create_null_income_features()

    try:
        # 过滤有效数据
        valid_qualifications = []
        invalid_qualifications = []
        for item in qualifications:
            income_info = item.get('informed_income', {})
            if income_info and income_info.get('amount') and item.get('update_datetime') and item.get('request_date'):
                has_data = 1
                freq = income_info.get('frequency', '')
                amt = float(income_info.get('amount', 0))
                currency = income_info.get('currency', '')
                date = income_info.get('date', '')
                update_time = item['update_datetime']
                request_date = item['request_date']

                # 计算更新时间差
                try:
                    # 计算更新时间差
                    fmt = "%Y-%m-%d"  # 根据你的实际格式调整
                    update_dt = datetime.strptime(update_time, fmt)
                    request_dt = datetime.strptime(request_date, fmt)

                    days_from_update = (request_dt - update_dt).days
                except:
                    days_from_update = None

                if days_from_update is not None:
                    valid_qualifications.append({
                        'has_data': has_data,
                        'days_from_last_update': days_from_update,
                        'amt': amt,
                        'currency': currency,
                        'freq': freq,
                        'date': date
                    })

        if not valid_qualifications:
            return create_null_income_features()

        
        # 选择最近更新的记录
        latest_record = min(valid_qualifications, key=lambda x: x['days_from_last_update'])

        return {
            "inst_income_has_data": latest_record['has_data'],
            "inst_income_days_from_last_update": latest_record['days_from_last_update'],
            "inst_income_amt": latest_record['amt'],
            "inst_income_currency": latest_record['currency'],
            "inst_income_freq": latest_record['freq'],
            "inst_income_date": latest_record['date'],
        }
    except Exception as e:
        print(f"提取收入特征时出错: {e}")
        return create_null_income_features()


def process_personal_data(raw_data, cpf, request_date):
    """
    处理个人数据，提取特征

    Args:
        raw_data: 原始个人数据JSON字符串
        cpf: CPF号码

    Returns:
        处理后的特征字典，包含demogr_fts和income_fts
    """
    if not raw_data:
        null_income = create_null_income_features()
        null_demogr = create_null_demogr_features()
        return {
            "income_fts": null_income,
            "demogr_fts": null_demogr
        }

    try:
        # 解析JSON数据
        if isinstance(raw_data, str):
            data = json.loads(raw_data)
        else:
            data = raw_data
        logger.info(f"raw_data: {data}")
        logger.info(f"Processing data for CPF: {cpf}")
        
        companies_data = data.get("get_open_finance_personal_registration_data", [])

        identifications = []
        qualifications = []

        # 遍历所有公司数据
        for company_data in companies_data:
            organisation_name = company_data.get('organisation_name', '')
            data_list = company_data.get('data', [])

            for item in data_list:
                url = item.get('url', '')

                # 解析API响应
                api_response = item.get('rsp', '')
                if isinstance(api_response, str):
                    try:
                        api_response = json.loads(api_response)
                    except:
                        continue

                rsp_data = api_response.get('data', [])

                # 根据URL类型解析不同的数据
                if 'identifications' in url:
                    identification = parse_identifications(json.dumps(rsp_data), request_date)
                    if identification:
                        identifications.append(identification)
                elif 'qualifications' in url:
                    qualification = parse_qualifications(json.dumps(rsp_data), request_date)
                    if qualification:
                        qualifications.append(qualification)

        # 提取特征
        income_result = extract_income_features(qualifications)
        demogr_result = extract_demogr_features(identifications)

        return {
            "income_fts": income_result,
            "demogr_fts": demogr_result
        }

    except Exception as e:
        print(f"处理个人数据时出错: {e}")
        null_income = create_null_income_features()
        null_demogr = create_null_demogr_features()
        return {
            "income_fts": null_income,
            "demogr_fts": null_demogr
        }



def process_row(raw_data, cpf, request_date):
        
    result = process_personal_data(raw_data, cpf, request_date)
    # 确保返回值格式正确
    income_fts = result.get("income_fts", create_null_income_features())
    demogr_fts = result.get("demogr_fts", create_null_demogr_features())
    # 将所有值转换为字符串以符合MapType定义
    income_fts_str = {k: str(v) if v is not None else None for k, v in income_fts.items()}
    demogr_fts_str = {k: str(v) if v is not None else None for k, v in demogr_fts.items()}

    feature_groups = {}
    feature_groups.update(income_fts_str)
    feature_groups.update(demogr_fts_str)

    return {
        "feature_group": feature_groups
    }




def process_dataframe(input_df):
    """
    处理输入DataFrame
    Args:
        input_df: 输入DataFrame，包含shopee_user_id, cpf, get_open_finance_personal_registration_data
    Returns:
        处理后的DataFrame，包含shopee_user_id, cpf, origin_data, feature_groups
    """
    logger.info("开始处理DataFrame...")
    # 检查输入数据
    input_count = input_df.count()
    logger.info(f"输入数据记录数: {input_count}")
    # 缓存输入DataFrame以提高性能
    input_df.cache()
    # 定义返回类型
    return_schema = StructType([
        StructField("feature_group", MapType(StringType(), StringType(), True), True)
    ])
    # 创建处理UDF
    process_udf = udf(process_row, return_schema)
    # 应用UDF处理数据，添加重分区以优化性能
    result_df = input_df.repartition(200).withColumn(
        "processed_result",
        process_udf(
            col("get_open_finance_personal_registration_data"),
            col("cpf"),
            col("request_date")
        )
    )
    # 展开处理结果
    final_df = result_df.select(
        col("flow_no"),
        col("shopee_user_id"),
        col("cpf"),
        col("request_date"),
        col("request_time"),
        to_json(col("processed_result.feature_group")).alias("feature_group"),
        col("get_open_finance_personal_registration_data").alias("origin_data"),
    )
    # 缓存结果DataFrame
    final_df.cache()
    # 检查输出数据质量
    output_count = final_df.count()
    logger.info(f"输出数据记录数: {output_count}")
    if output_count != input_count:
        logger.warning(f"输入输出记录数不匹配！输入: {input_count}, 输出: {output_count}")
    # 检查空值情况
    null_cpf_count = final_df.filter(col("cpf").isNull() | (col("cpf") == "")).count()
    null_feature_count = final_df.filter(col("feature_group").isNull()).count()
    logger.info(
        f"数据质量检查 - CPF空值: {null_cpf_count}, feature_groups: {null_feature_count}")
    return final_df

def read_source_data(ds: str, source_table: str = "risk.dws_risk_br_openfinance_request_df"):
    """
    从源表读取数据
    Args:
        source_table: 源表名称
    Returns:
        DataFrame包含所需的列
    """
    df = spark.table(source_table)
    # 选择所需的列
    selected_df = df.select(
        col("flow_no"),
        col("shopee_user_id"),
        col("cpf"),
        col("request_date"),
        col("request_time"),
        col("get_open_finance_personal_registration_data")
    ).filter(
        col("get_open_finance_personal_registration_data").isNotNull() &
        (col("get_open_finance_personal_registration_data") != "") &
        (col("grass_date") == ds)
    )
    return selected_df

def write_to_hive(spark_df, category='', schema="dev_risk", table="ads_risk_br_openfinance_df", env="",
                  ds="2025-09-17"):
    """
    将 Spark DataFrame 写入 Hive 表 (支持分区覆盖)
    """
    root_path = f"/projects/risk/hdfs/{env}/risk_mart/ads/ads_risk_openfinance_request_df/grass_region=BR"
    # 添加分区字段 grass_date
    spark_df_with_partition = spark_df \
        .withColumn("grass_date", F.lit(ds)) \
        .withColumn("category", F.lit(category))
    # 覆盖写入 Hive (支持分区)
    (
        spark_df_with_partition
        .write.mode("overwrite")
        .format("parquet")
        .partitionBy("category", "grass_date")
        .option("partitionOverwriteMode", "dynamic")
        .option("parquet.compression", "snappy")  # 避免 zstd 兼容问题
        .option("spark.sql.parquet.writeLegacyFormat", "true")  # 避免 decimal/时间戳兼容
        .parquet(f"{root_path}")
    )
    print(f"✅ Data written to Hive table: {schema}.{table} partition grass_date={ds}")
    print(f"📊 Total records written: {spark_df_with_partition.count()}")
    # 如果 Hive Metastore 需要同步 HDFS 分区，可以在 Spark SQL 里调用 MSCK
    spark.sql(f"MSCK REPAIR TABLE {schema}.{table}")
    

In [2]:
grass_date = '${ds}'
df = read_source_data(grass_date)

In [4]:
final_df = process_dataframe(df)
write_to_hive(final_df, category='personal_profile', schema="${SCHEMA}", table="ads_risk_br_openfinance_request_df", env="${ENV}", ds=grass_date)

✅ Data written to Hive table: dev_risk.ads_risk_br_openfinance_request_df partition grass_date=2025-11-12
📊 Total records written: 1041471